In [45]:
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
from tqdm import tqdm

In [29]:
train = pd.read_csv("dataset/train_data.csv")
test = pd.read_csv("dataset/test_data.csv")

print(train.head())
print(test.head())

     pair_id                                          py_source  \
0  pair_0000  def f_gold ( num , divisor ) :\n    while ( nu...   
1  pair_0001  def f_gold ( arr , n ) :\n    mp = dict ( )\n ...   
2  pair_0002  def f_gold ( s ) :\n    length = len ( s )\n  ...   
3  pair_0003  def f_gold ( arr , n ) :\n    arr.sort ( ) ;\n...   
4  pair_0004  import sys\n\ndef f_gold ( arr , n ) :\n    re...   

                                          cpp_source  
0  using namespace std;\nint f_gold ( int num, in...  
1  using namespace std;\nint f_gold ( int arr [ ]...  
2  using namespace std;\nint f_gold ( string str ...  
3  using namespace std;\nint f_gold ( int arr [ ]...  
4  using namespace std;\nint f_gold ( int arr [ ]...  
    type          datapointID  \
0  query  py_test_public_0000   
1  query  py_test_public_0001   
2  query  py_test_public_0002   
3  query  py_test_public_0003   
4  query  py_test_public_0004   

                                              source  
0  def f_gold

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self):
        super().__init__()

        self.py = tokenizer(
            train["py_source"].tolist(), 
            padding=True,
            truncation=True,
            return_tensors="pt"
            )
        
        self.cpp = tokenizer(
            train["cpp_source"].tolist(), 
            padding=True,
            truncation=True,
            return_tensors="pt"
            )
        
    def __len__(self):
        return len(self.py)
    
    def __getitem__(self, index):
        return self.py["input_ids"][index], self.py["attention_mask"][index], self.cpp["input_ids"][index], self.cpp["attention_mask"][index],

dataset = TextDataset()
loader = DataLoader(dataset, batch_size=8)

In [ ]:
class CodeEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("bert-base-uncased")

    def mean_pool(self, out, mask):
        token_emb = out.last_hidden_state
        mask = mask.unsqueeze(-1)
        return (token_emb * mask).sum(1) / mask.sum(1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        emb = self.mean_pool(out, attention_mask)
        emb = F.normalize(emb, p=2, dim=1)

        return emb

In [42]:
device = "mps"
model = CodeEmbedder().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(50):
    total_loss = 0

    model.train()
    for py_ids, py_att, cpp_ids, cpp_att in loader:
        py_ids = py_ids.to(device)
        py_att = py_att.to(device)

        cpp_ids = cpp_ids.to(device)
        cpp_att = cpp_att.to(device)

        py_emb = model(py_ids, py_att)
        cpp_emb = model(cpp_ids, cpp_att)

        logits = py_emb @ cpp_emb.T
        labels = torch.arange(logits.size(0), device=device)

        loss1 = F.cross_entropy(logits, labels)
        loss2 = F.cross_entropy(logits.T, labels)

        loss = (loss1 + loss2) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(epoch, total_loss / len(loader))
        

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22949.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0 1.0565626621246338
1 0.9991391897201538
2 0.9465813636779785
3 0.8688365817070007
4 0.7788402438163757
5 0.7261753082275391
6 0.667973518371582
7 0.6245822906494141
8 0.5799577236175537
9 0.5397500991821289
10 0.5136047601699829
11 0.4907139539718628
12 0.47020718455314636
13 0.44347167015075684
14 0.430652916431427
15 0.42250025272369385
16 0.4151480197906494
17 0.4114444851875305
18 0.4046992063522339
19 0.40121904015541077
20 0.3970620632171631
21 0.39385658502578735
22 0.39003121852874756
23 0.3881269693374634
24 0.3871099054813385
25 0.3856194019317627
26 0.384940505027771
27 0.3827539086341858
28 0.38051837682724
29 0.37922656536102295
30 0.37806421518325806
31 0.3781026005744934
32 0.3770512342453003
33 0.37656712532043457
34 0.3765442967414856
35 0.3753241300582886
36 0.3754844069480896
37 0.37439727783203125
38 0.3744509816169739
39 0.3740968704223633
40 0.3736186623573303
41 0.373390257358551
42 0.3738132119178772
43 0.3734923005104065
44 0.3732306957244873
45 0.37276145815

In [ ]:
queries = test[test["type"] == "query"]
candidates = test[test["type"] == "candidate"]

class CppDataset(Dataset):
    def __init__(self):
        super().__init__()

        self.cpp = tokenizer(
            candidates["source"].tolist(), 
            padding=True,
            truncation=True,
            return_tensors="pt"
            )
        
    def __len__(self):
        return len(self.py)
    
    def __getitem__(self, index):
        return self.cpp["input_ids"][index], self.cpp["attention_mask"][index],

cpp_loader = DataLoader(CppDataset(), batch_size=8)

cpp_embs = []

model.eval()

with torch.no_grad():
    for batch in cpp_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        emb = model(**batch)
        cpp_embs.append(emb.cpu())

cpp_embs = torch.cat(cpp_embs)

py_emb = model(**py_query_enc).cpu()
scores = py_emb @ cpp_embs.T
ranked_idx = scores.argsort(dim=1, descending=True)
        

100%|██████████| 100/100 [00:00<00:00, 383041.46it/s]
